# AxioScan CZI → petro-image

This notebook is a small testing interface for the standalone pipeline. It inspects the inputs first, converts them with the same command-line program used on a server, and summarizes the generated library.

The output directory must not already exist. This protects completed conversions from accidental replacement.

## 1. Locate the repository and check dependencies

In [1]:
from pathlib import Path
import json
import subprocess
import sys

from IPython.display import JSON, display

repo_root = Path.cwd().resolve()
if repo_root.name == "czi_pipeline":
    repo_root = repo_root.parent
if not (repo_root / "czi_pipeline" / "batch.py").is_file():
    raise RuntimeError(
        "Start Jupyter in the petro-image repository or its czi_pipeline folder."
    )

for package_name in ("pylibCZIrw", "numpy", "PIL"):
    try:
        __import__(package_name)
    except ImportError as error:
        raise RuntimeError(
            "Install the pipeline dependencies in this notebook's Python environment:\n"
            f"{sys.executable} -m pip install -r "
            f"{repo_root / 'czi_pipeline' / 'requirements.txt'}"
        ) from error

print(f"Repository: {repo_root}")
print(f"Python:     {sys.executable}")

Repository: /Users/glennsharman/Documents/GitHub/petro-image
Python:     /opt/anaconda3/envs/petro-czi/bin/python


## 2. Choose inputs and conversion settings

`INPUT_PATH` may be one CZI file or a directory. Choose a new `OUTPUT_PATH` each time you run a test.

In [2]:
INPUT_PATH = Path("/path/to/czi-files")
INPUT_PATH = Path('/Users/glennsharman/Desktop/Test Project/czi')
OUTPUT_PATH = Path("/path/to/new-output")
OUTPUT_PATH = Path('/Users/glennsharman/Desktop/Test Project/czi/Test conversion')

DOWNSAMPLE = 0.25       # 1, 0.5, 0.25, 0.125, ...
JPEG_QUALITY = 90      # 1 through 100
GROUP = "Imported CZI"
RECURSIVE = False

INPUT_PATH = INPUT_PATH.expanduser().resolve()
OUTPUT_PATH = OUTPUT_PATH.expanduser().resolve()
print(f"Input:  {INPUT_PATH}")
print(f"Output: {OUTPUT_PATH}")

Input:  /Users/glennsharman/Desktop/Test Project/czi
Output: /Users/glennsharman/Desktop/Test Project/czi/Test conversion


## 3. Inspect and validate the scans

This reads metadata only. It verifies the supported AxioScan profile and confirms that calibrated X and Y pixel dimensions are present and equal.

In [3]:
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from czi_pipeline.batch import find_sources, inspect_source, sample_id_for_key, canonical_sample_key

sources = find_sources(INPUT_PATH, RECURSIVE)
if not sources:
    raise ValueError(f"No CZI files found at {INPUT_PATH}")

inspection_rows = []
for source in sources:
    inspection = inspect_source(source)
    pixel_size = inspection["pixelSizeMicrometers"]
    inspection_rows.append({
        "file": source.name,
        "sampleId": sample_id_for_key(canonical_sample_key(source)),
        "width": inspection["bounds"]["width"],
        "height": inspection["bounds"]["height"],
        "channels": len(inspection["channels"]),
        "xMicrometersPerPixel": pixel_size["x"],
        "yMicrometersPerPixel": pixel_size["y"],
        "squarePixels": inspection["hasSquarePixels"],
    })

display(JSON(inspection_rows, expanded=True))

<IPython.core.display.JSON object>

## 4. Run the conversion

Conversion progress is printed below the cell. A failure removes the temporary batch output rather than publishing a partial library.

In [5]:
if OUTPUT_PATH.exists():
    raise FileExistsError(
        f"Output already exists: {OUTPUT_PATH}\nChoose a new path; the notebook will not delete it."
    )

command = [
    sys.executable,
    str(repo_root / "czi_pipeline" / "batch.py"),
    str(INPUT_PATH),
    str(OUTPUT_PATH),
    "--downsample", str(DOWNSAMPLE),
    "--quality", str(JPEG_QUALITY),
    "--group", GROUP,
]
if RECURSIVE:
    command.append("--recursive")

print("Running:", " ".join(command))
process = subprocess.Popen(
    command,
    cwd=repo_root,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)
for line in process.stdout:
    print(line, end="")
return_code = process.wait()
if return_code:
    raise RuntimeError(f"Conversion exited with status {return_code}")

Running: /opt/anaconda3/envs/petro-czi/bin/python /Users/glennsharman/Documents/GitHub/petro-image/czi_pipeline/batch.py /Users/glennsharman/Desktop/Test Project/czi /Users/glennsharman/Desktop/Test Project/czi/Test conversion --downsample 0.25 --quality 90 --group Imported CZI
Inspecting EXT-Malkowski-EppersonPhD-IODP_05-redo.czi
Inspecting RT-MR-S.czi
Converting EXT-Malkowski-EppersonPhD-IODP_05-redo.czi [b8tcg9jn]
Converting RT-MR-S.czi [b9r0g7p0]
{"libraryPath": "/Users/glennsharman/Desktop/Test Project/czi/Test conversion/library.json", "outputPath": "/Users/glennsharman/Desktop/Test Project/czi/Test conversion", "sampleCount": 2}


## 5. Review the generated library

In [ ]:
library_path = OUTPUT_PATH / "library.json"
library = json.loads(library_path.read_text(encoding="utf-8"))

summary = []
for sample in library["samples"]:
    tiles = [
        tile
        for tile_set in sample.get("tileSets", [])
        for tile in tile_set.get("tiles", [])
    ]
    missing = [tile["uri"] for tile in tiles if not (OUTPUT_PATH / tile["uri"]).is_file()]
    summary.append({
        "sampleId": sample["sampleId"],
        "title": sample["title"],
        "pixelsPerMeter": sample.get("pixelsPerMeter"),
        "tileSets": len(sample.get("tileSets", [])),
        "DZI descriptors": len(tiles),
        "missing descriptors": missing,
    })

print(f"Library: {library_path}")
display(JSON(summary, expanded=True))